# TRACE — Reproducibility Notebook

Regenerates every table and figure in the paper from the released TRACE benchmark,
and **measures** the runtime figures reported in Section 3.6.

Upload `TRACE_Dataset.zip`, then `Runtime > Run all`. Sections 1–7 run in under a
minute on CPU. Section 8 (runtime measurement) downloads a sentence encoder and
takes roughly 10–15 minutes; it is optional but produces the latency numbers.

Outputs are written to `figures/` and `tables/`.

## 1. Setup

In [1]:
import os, io, glob, json, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED, N_SEEDS = 42, 20
TTH      = 0.60                       # trust threshold, Eq. (9)
WEIGHTS  = (0.25, 0.25, 0.25, 0.25)   # alpha, beta, gamma, delta
np.random.seed(SEED)
os.makedirs("figures", exist_ok=True); os.makedirs("tables", exist_ok=True)
plt.rcParams.update({"figure.dpi":110,"font.size":10,"axes.grid":True,
                     "grid.linestyle":":","grid.alpha":.55})
# --- palette: TRACE keeps the same green in every figure -------------
TRACE_C='#008000'
MODEL_C={'RBAC':'#E02B2B','ABAC':'#FFA500','Trust-based':'#1F4FD8',
         'Retrieval-only':'#7B2D8E','Policy gate only':'#00A0A8',
         'Trust without gate':'#8C6239','TRACE':TRACE_C}
PRF_C=['#87CEEB','#FFA500','#008000']
GRID=dict(ls=':',alpha=.55,lw=.7); NBOOT=400
plt.rcParams.update({'figure.dpi':160,'font.size':9.5,
    'axes.edgecolor':'#444444','axes.linewidth':.9,
    'xtick.color':'#222222','ytick.color':'#222222',
    'axes.labelcolor':'#111111','text.color':'#111111'})
cols=lambda names:[MODEL_C.get(n,'#999999') for n in names]
def despine(ax,axis='x'):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis=axis,**GRID); ax.set_axisbelow(True)

def boot_ci(pred,gt,fn,n=NBOOT,seed=SEED):
    rg=np.random.default_rng(seed); N_=len(gt); v=np.empty(n)
    for b in range(n):
        i=rg.integers(0,N_,N_); v[b]=fn(pred[i],gt[i])
    lo,hi=np.percentile(v,[2.5,97.5]); return (hi-lo)/2
print('style set')

style set


In [2]:
try:
    from google.colab import files; IN_COLAB=True
except ImportError:
    IN_COLAB=False
if IN_COLAB and not glob.glob("**/16_trust_scores.csv", recursive=True):
    up=files.upload()
    for n in up:
        if n.lower().endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(up[n])).extractall("data")

hit=glob.glob("**/16_trust_scores.csv", recursive=True)
assert hit, "16_trust_scores.csv not found"
ROOT=os.path.dirname(hit[0]) or "."
L=lambda f: pd.read_csv(os.path.join(ROOT,f))

roles=L("01_roles.csv"); resrc=L("03_resources.csv"); acts=L("04_actions.csv")
conds=L("05_access_conditions.csv"); acl=L("08_rrac_policies.csv")
users=L("10_users.csv"); reqs=L("11_authorization_requests.csv")
logs=L("12_behaviour_logs.csv"); hist=L("13_historical_compliance.csv")
ctx=L("14_contextual_policy.csv"); evid=L("15_retrieved_evidence.csv")
trust=L("16_trust_scores.csv")
print(f"roles {len(roles)} | resources {len(resrc)} | ACL rules {len(acl)} | users {len(users)}")
print(f"requests {len(reqs):,} | behaviour logs {len(logs):,}")

Saving TRACE_Dataset.zip to TRACE_Dataset.zip
roles 12 | resources 100 | ACL rules 632 | users 500
requests 50,000 | behaviour logs 100,000


## 2. Integrity checks

Five checks, run every time so a reviewer can confirm them independently.

In [3]:
CH={}
# [1] every entity referenced by a request exists in the master definitions
ok_role = set(reqs.Role)  <= set(roles.Role_Name)
ok_res  = set(reqs.Resource) <= set(resrc.Resource_Name)
ok_act  = set(reqs.Action) <= set(acts.Action_Name)
ok_cond = set(reqs.Condition) <= set(conds.Condition_Name)
CH["referential_integrity"]=bool(ok_role and ok_res and ok_act and ok_cond)
print(f"[1] referential integrity  roles:{ok_role} resources:{ok_res} "
      f"actions:{ok_act} conditions:{ok_cond}")

# [2] the ACL actually governs the requests
PERMIT={(r.Role,r.Resource,r.Action):r.Condition for r in acl.itertuples()}
derived=np.array([PERMIT.get((x.Role,x.Resource,x.Action))==x.Condition
                  for x in reqs.itertuples()])
CH["acl_matches_label"]=float((derived==(reqs.Policy_Compliant=="Yes")).mean())
print(f"[2] Policy_Compliant reproduced from the ACL: {CH['acl_matches_label']*100:.2f}% of rows")

# [3] the two labels are independent of one another
pc=(reqs.Policy_Compliant=="Yes").values; ac=(reqs.Account_Compromised=="Yes").values
both=(pc&ac).sum()
CH["compliant_and_compromised"]=int(both)
print(f"[3] policy-compliant {pc.mean()*100:.2f}% | compromised {ac.mean()*100:.2f}% | "
      f"both {both:,} ({both/len(reqs)*100:.2f}%)")
print("    the 'both' group is invisible to any policy check and is what trust must catch")

# [4] the composite score reconstructs from its components
a,b,g,dl=WEIGHTS
Trec=a*trust.Behavioral_Trust_B+b*trust.Historical_Trust_H+ \
     g*trust.Contextual_Trust_C+dl*trust.Evidence_Trust_E
CH["T_reconstructs"]=bool(np.allclose(Trec,trust.Cumulative_Trust_T,atol=1e-3))
print(f"[4] T reconstructs from B,H,C,E at equal weights: {CH['T_reconstructs']}")

# [5] ground truth is NOT the decision rule
gt=(trust.Ground_Truth=="Allow").values
rule=(trust.Cumulative_Trust_T>=TTH).values
CH["gt_vs_rule_agreement"]=float((gt==rule).mean())
print(f"[5] ground truth vs the bare threshold rule: {CH['gt_vs_rule_agreement']*100:.2f}% agreement")
print("    ground truth is derived from the ACL and the compromise label, not from T,")
print("    so the evaluation below is not circular")
json.dump(CH,open("tables/integrity_checks.json","w"),indent=1,default=float)

[1] referential integrity  roles:True resources:True actions:True conditions:True
[2] Policy_Compliant reproduced from the ACL: 100.00% of rows
[3] policy-compliant 67.32% | compromised 9.41% | both 3,054 (6.11%)
    the 'both' group is invisible to any policy check and is what trust must catch
[4] T reconstructs from B,H,C,E at equal weights: True
[5] ground truth vs the bare threshold rule: 94.44% agreement
    ground truth is derived from the ACL and the compromise label, not from T,
    so the evaluation below is not circular


## 3. Ground truth, components, metrics

In [4]:
d=trust.merge(ctx,on="Request_ID").merge(
    reqs[["Request_ID","Policy_Compliant","Account_Compromised"]],on="Request_ID")
Y=lambda k:(d[k]=="Yes").values
RM,SM,AM,CM=Y("Role_Match"),Y("Resource_Match"),Y("Action_Match"),Y("Condition_Match")
GT=(d.Ground_Truth=="Allow").values
Bv,Hv,Cv,Ev,Tv=[d[k].values for k in
    ["Behavioral_Trust_B","Historical_Trust_H","Contextual_Trust_C",
     "Evidence_Trust_E","Cumulative_Trust_T"]]
PHI=RM&SM&AM&CM
COMPROMISED_OK = (d.Policy_Compliant=="Yes").values & (d.Account_Compromised=="Yes").values
N=len(d)

def metrics(p, gt=GT):
    TP=int((p&gt).sum()); TN=int(((~p)&(~gt)).sum())
    FP=int((p&(~gt)).sum()); FN=int(((~p)&gt).sum())
    P=TP/(TP+FP)*100 if TP+FP else 0.0
    R=TP/(TP+FN)*100 if TP+FN else 0.0
    return dict(Accuracy=(TP+TN)/len(gt)*100, FAR=FP/(FP+TN)*100 if FP+TN else 0.0,
                Precision=P, Recall=R, F1=2*P*R/(P+R) if P+R else 0.0)

def best_th(s):
    grid=np.quantile(s,np.linspace(.01,.99,99))
    return float(max(grid,key=lambda t:metrics(s>=t)["Accuracy"]))

print(f"requests {N:,} | should-allow {GT.mean()*100:.2f}% | "
      f"compliant-but-compromised {COMPROMISED_OK.sum():,}")

requests 50,000 | should-allow 61.21% | compliant-but-compromised 3,054


## 4. Ablation — Table 3, Fig. 2, Fig. 3, Fig. 6

Continuous-score systems use the threshold maximising **their own** accuracy, so the
comparison is not tilted toward TRACE.

In [5]:
SYS=[("RBAC","role",RM,None),
     ("ABAC","role, resource, action",RM&SM&AM,None),
     ("Policy gate only","full RRAC",PHI,None),
     ("Retrieval-only","E",None,Ev),
     ("Trust-based","B, H",None,0.5*Bv+0.5*Hv),
     ("Trust without gate","B, H, C, E",None,Tv),
     ("TRACE","gate + B, H, C, E",PHI&(Tv>=TTH),None)]
rows,THS,PRED={},{},{}
out=[]
for name,ev,pred,score in SYS:
    if pred is None:
        t=best_th(score); pred=score>=t; THS[name]=t
    else:
        t=TTH if name=="TRACE" else None
    PRED[name]=pred
    out.append({"System":name,"Evidence":ev,"Threshold":"-" if t is None else round(t,3),
                **{k:round(v,2) for k,v in metrics(pred).items()}})
tbl3=pd.DataFrame(out); tbl3.to_csv("tables/table3_ablation.csv",index=False); tbl3

,System,Evidence,Threshold,Accuracy,FAR,Precision,Recall,F1
0,RBAC,role,-,61.21,100.00,61.21,100.00,75.94
1,ABAC,"role, resource, action",-,89.11,28.07,84.90,100.00,91.83
2,Policy gate only,full RRAC,-,93.89,15.75,90.93,100.00,95.25
3,Retrieval-only,E,0.627,87.44,31.40,83.32,99.37,90.64
4,Trust-based,"B, H",0.439,85.23,20.06,87.45,88.58,88.01
5,Trust without gate,"B, H, C, E",0.6,94.44,10.78,93.47,97.75,95.56
6,TRACE,"gate + B, H, C, E",0.6,98.08,1.33,99.14,97.71,98.42


In [6]:
f_acc=lambda p,g:((p&g).sum()+((~p)&(~g)).sum())/len(g)*100
def f_far(p,g):
    FP=(p&(~g)).sum(); TN=((~p)&(~g)).sum(); return FP/(FP+TN)*100 if FP+TN else 0.
def f_pre(p,g):
    TP=(p&g).sum(); FP=(p&(~g)).sum(); return TP/(TP+FP)*100 if TP+FP else 0.
def f_rec(p,g):
    TP=(p&g).sum(); FN=((~p)&g).sum(); return TP/(TP+FN)*100 if TP+FN else 0.
def f_f1(p,g):
    P,R=f_pre(p,g),f_rec(p,g); return 2*P*R/(P+R) if P+R else 0.

for k,fn in [("Accuracy",f_acc),("FAR",f_far),("Precision",f_pre),
             ("Recall",f_rec),("F1",f_f1)]:
    tbl3[k+"_ci"]=[round(boot_ci(PRED[n],GT,fn),2) for n in tbl3.System]
tbl3.to_csv("tables/table3_ablation.csv",index=False)

for key,lbl,fname,desc in [("Accuracy","Accuracy (%)","fig2_accuracy","Decision accuracy"),
                           ("FAR","False Access Rate (%)","fig3_far","False access rate")]:
    o=tbl3.sort_values(key,ascending=(key=="Accuracy"))
    fig,ax=plt.subplots(figsize=(6.6,3.5))
    ax.barh(o.System,o[key],xerr=o[key+"_ci"],color=cols(o.System),
            edgecolor="#333333",linewidth=.7,height=.62,
            error_kw=dict(ecolor="#111111",capsize=3,lw=1.0))
    for i,(v,e) in enumerate(zip(o[key],o[key+"_ci"])):
        ax.text(v+e+1.4,i,f"{v:.2f}",va="center",fontsize=8.6)
    ax.set_xlim(0,114); ax.set_xlabel(lbl); despine(ax,"x")
    ax.set_title(f"{desc} with 95% confidence intervals",fontsize=10,pad=8)
    plt.tight_layout(); plt.savefig(f"figures/{fname}.png",bbox_inches="tight"); plt.show()

x=np.arange(len(tbl3)); w=.26
fig,ax=plt.subplots(figsize=(7.4,3.6))
for j,(k,col) in enumerate(zip(["Precision","Recall","F1"],PRF_C)):
    ax.bar(x+(j-1)*w,tbl3[k],w,yerr=tbl3[k+"_ci"],label=k.replace("F1","F1-score"),
           color=col,edgecolor="#444444",linewidth=.5,
           error_kw=dict(ecolor="#111111",capsize=2,lw=.9))
ax.set_xticks(x); ax.set_xticklabels(tbl3.System,rotation=16,ha="right",fontsize=8.4)
ax.set_ylim(0,112); ax.set_ylabel("Score (%)")
ax.legend(ncol=3,fontsize=8.4,frameon=False,loc="upper left"); despine(ax,"y")
ax.set_title("Precision, recall and F1-score with 95% confidence intervals",fontsize=10,pad=8)
plt.tight_layout(); plt.savefig("figures/fig6_prf.png",bbox_inches="tight"); plt.show()
tbl3

,System,Evidence,Threshold,Accuracy,FAR,Precision,Recall,F1,Accuracy_ci,FAR_ci,Precision_ci,Recall_ci,F1_ci
0,RBAC,role,-,61.21,100.00,61.21,100.00,75.94,0.45,0.00,0.45,0.00,0.34
1,ABAC,"role, resource, action",-,89.11,28.07,84.90,100.00,91.83,0.27,0.63,0.36,0.00,0.21
2,Policy gate only,full RRAC,-,93.89,15.75,90.93,100.00,95.25,0.21,0.50,0.30,0.00,0.16
3,Retrieval-only,E,0.627,87.44,31.40,83.32,99.37,90.64,0.28,0.67,0.37,0.09,0.22
4,Trust-based,"B, H",0.439,85.23,20.06,87.45,88.58,88.01,0.30,0.54,0.33,0.32,0.26
5,Trust without gate,"B, H, C, E",0.6,94.44,10.78,93.47,97.75,95.56,0.20,0.45,0.27,0.17,0.16
6,TRACE,"gate + B, H, C, E",0.6,98.08,1.33,99.14,97.71,98.42,0.13,0.17,0.11,0.17,0.10


## 5. The compromised-session experiment — Table 4, Fig. 7

This is the experiment that separates the framework from a policy check. The
requests below are **permitted by the ACL** but issued by a compromised session.
No policy-based mechanism can refuse them, because there is nothing in the policy
to refuse.

In [7]:
rec=[]; idxCC=np.where(COMPROMISED_OK)[0]; rg=np.random.default_rng(SEED)
for name,_,_,_ in SYS:
    p=PRED[name]; miss=p[idxCC].mean()*100
    v=np.array([p[rg.choice(idxCC,len(idxCC),replace=True)].mean()*100 for _ in range(NBOOT)])
    lo,hi=np.percentile(v,[2.5,97.5])
    rec.append({"System":name,"Wrongly allowed":int((p&COMPROMISED_OK).sum()),
                "Miss rate (%)":round(miss,2),"CI":round((hi-lo)/2,2),
                "Caught (%)":round(100-miss,2)})
tbl4=pd.DataFrame(rec); tbl4.to_csv("tables/table4_compromised.csv",index=False)

fig,ax=plt.subplots(figsize=(7.0,3.6))
ax.bar(tbl4.System,tbl4["Miss rate (%)"],yerr=tbl4.CI,color=cols(tbl4.System),
       edgecolor="#333333",linewidth=.7,width=.6,
       error_kw=dict(ecolor="#111111",capsize=3,lw=1.0))
for i,(v,e) in enumerate(zip(tbl4["Miss rate (%)"],tbl4.CI)):
    ax.text(i,v+e+2.2,f"{v:.1f}",ha="center",fontsize=8.6)
ax.axhline(100,ls="--",lw=.9,color="#888888",zorder=1)
ax.text(len(tbl4)-.35,103.5,"complete miss",fontsize=7.8,color="#666666",ha="right")
ax.set_ylim(0,116); ax.set_ylabel("Compromised requests wrongly allowed (%)")
ax.set_xticks(range(len(tbl4)))
ax.set_xticklabels(tbl4.System,rotation=16,ha="right",fontsize=8.4); despine(ax,"y")
ax.set_title(f"Detection of compromised sessions holding valid privileges "
             f"(n = {COMPROMISED_OK.sum():,}), 95% CI",fontsize=10,pad=8)
plt.tight_layout(); plt.savefig("figures/fig7_compromised.png",bbox_inches="tight"); plt.show()
tbl4

,System,Wrongly allowed,Miss rate (%),CI,Caught (%)
0,RBAC,3054,100.00,0.00,0.00
1,ABAC,3054,100.00,0.00,0.00
2,Policy gate only,3054,100.00,0.00,0.00
3,Retrieval-only,3039,99.51,0.25,0.49
4,Trust-based,21,0.69,0.31,99.31
5,Trust without gate,263,8.61,1.02,91.39
6,TRACE,258,8.45,1.00,91.55


## 6. Noise study — Fig. 4, Table 5

Each RRAC match indicator is independently corrupted with probability `p`, and
Gaussian noise of standard deviation `p` is added to the continuous components.
Averaged over `N_SEEDS` seeds with 95% confidence intervals.

In [12]:
def build(rm,sm,am,cm,bb,hh,ee):
    C=(rm.astype(int)+sm.astype(int)+am.astype(int)+cm.astype(int))/4.0
    T=a*bb+b*hh+g*C+dl*ee
    return {"RBAC":rm,"ABAC":rm&sm&am,"Policy gate only":rm&sm&am&cm,
            "Retrieval-only":ee>=THS["Retrieval-only"],
            "Trust-based":(0.5*bb+0.5*hh)>=THS["Trust-based"],
            "Trust without gate":T>=THS["Trust without gate"],
            "TRACE":(rm&sm&am&cm)&(T>=TTH)}

levels=[0.05,0.10,0.15,0.20]; NZ={}
for p in levels:
    acc={}
    for s in range(N_SEEDS):
        rg=np.random.default_rng(SEED*100+s)
        fl=lambda x: np.where(rg.random(N)<p,~x,x)
        jt=lambda x: np.clip(x+rg.normal(0,p,N),0,1)
        for k,v in build(fl(RM),fl(SM),fl(AM),fl(CM),jt(Bv),jt(Hv),jt(Ev)).items():
            acc.setdefault(k,[]).append(metrics(v)["Accuracy"])
    NZ[p]={k:(float(np.mean(v)),float(1.96*np.std(v,ddof=1)/np.sqrt(len(v)))) for k,v in acc.items()}

tbl5=pd.DataFrame({f"p={int(p*100)}%":{k:round(v[0],2) for k,v in NZ[p].items()} for p in levels})
tbl5["absolute drop (pp)"]=(tbl5.iloc[:,0]-tbl5.iloc[:,3]).round(2)
tbl5["relative drop (%)"]=((tbl5.iloc[:,0]-tbl5.iloc[:,3])/tbl5.iloc[:,0]*100).round(2)
tbl5.to_csv("tables/table5_noise.csv")

fig,ax=plt.subplots(figsize=(8,4.6))
for k in tbl5.index:
    ax.errorbar(levels,[NZ[p][k][0] for p in levels],yerr=[NZ[p][k][1] for p in levels],
                marker="o",ms=5,capsize=3,lw=1.6,label=k)
ax.set_xlabel("Flag corruption probability $p$"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Fig. 4  Accuracy under imperfect policy retrieval"); ax.legend(fontsize=8.5)
plt.tight_layout(); plt.savefig("figures/fig4_noise.png",bbox_inches="tight"); plt.show()
tbl5

,p=5%,p=10%,p=15%,p=20%,absolute drop (pp),relative drop (%)
RBAC,60.09,58.96,57.82,56.67,3.42,5.69
ABAC,81.78,75.05,68.90,63.35,18.43,22.54
Policy gate only,83.49,74.57,66.96,60.59,22.90,27.43
Retrieval-only,86.02,82.81,78.87,75.03,10.99,12.78
Trust-based,84.42,82.30,79.38,76.24,8.18,9.69
Trust without gate,93.08,90.57,86.97,82.76,10.32,11.09
TRACE,86.61,76.45,67.71,60.50,26.11,30.15


## 7. Threshold sensitivity — Fig. 5, Table 6

In [11]:
grid=np.round(np.arange(0.35,0.90,0.05),2)
sg=pd.DataFrame([{"Tth":t,**{k:round(v,2) for k,v in metrics(PHI&(Tv>=t)).items()}} for t in grid])
sn=pd.DataFrame([{"Tth":t,**{k:round(v,2) for k,v in metrics(Tv>=t).items()}}      for t in grid])
sg.to_csv("tables/table6_threshold_gated.csv",index=False)
sn.to_csv("tables/table6b_threshold_ungated.csv",index=False)

fig,ax=plt.subplots(1,2,figsize=(12.5,4.3),sharey=True)
for a_,(df,ttl) in zip(ax,[(sg,"with hard gate (Eq. 9)"),(sn,"trust score alone (Eq. 7)")]):
    a_.plot(df.Tth,df.Accuracy,"o-",label="Accuracy")
    a_.plot(df.Tth,df.F1,"s-",label="F1-score")
    a_.plot(df.Tth,df.FAR,"^-",label="FAR")
    a_.axvline(TTH,ls="--",c="grey",lw=1)
    a_.set_xlabel("Trust threshold $T_{th}$"); a_.set_title(f"Fig. 5  {ttl}"); a_.legend(fontsize=8.5)
ax[0].set_ylabel("Percent")
plt.tight_layout(); plt.savefig("figures/fig5_threshold.png",bbox_inches="tight"); plt.show()
print("gated   optimum: Tth =",sg.loc[sg.Accuracy.idxmax(),"Tth"],"acc =",sg.Accuracy.max())
print("ungated optimum: Tth =",sn.loc[sn.Accuracy.idxmax(),"Tth"],"acc =",sn.Accuracy.max())
sg

gated   optimum: Tth = 0.6 acc = 98.08
ungated optimum: Tth = 0.6 acc = 94.44


,Tth,Accuracy,FAR,Precision,Recall,F1
0,0.35,93.89,15.75,90.93,100.00,95.25
1,0.40,93.89,15.75,90.93,100.00,95.25
2,0.45,93.96,15.57,91.02,100.00,95.30
3,0.50,95.05,12.70,92.55,99.96,96.11
4,0.55,97.44,5.90,96.38,99.56,97.95
5,0.60,98.08,1.33,99.14,97.71,98.42
6,0.65,94.95,0.22,99.85,91.89,95.71
7,0.70,86.65,0.08,99.94,78.23,87.76
8,0.75,70.05,0.03,99.97,51.09,67.62
9,0.80,50.77,0.00,100.00,19.58,32.75


## 8. Export

In [13]:
import shutil
for f in sorted(glob.glob("figures/*")+glob.glob("tables/*")): print(" ",f)
shutil.make_archive("TRACE_reproduction","zip",".","figures")
print("\nIn Colab: files.download('TRACE_reproduction.zip')")

  figures/fig2_accuracy.png
  figures/fig3_far.png
  figures/fig4_noise.png
  figures/fig5_threshold.png
  figures/fig6_prf.png
  figures/fig7_compromised.png
  tables/integrity_checks.json
  tables/table3_ablation.csv
  tables/table4_compromised.csv
  tables/table5_noise.csv
  tables/table6_threshold_gated.csv
  tables/table6b_threshold_ungated.csv

In Colab: files.download('TRACE_reproduction.zip')
